In [8]:
from opensoundscape.annotations import BoxedAnnotations
import pandas as pd

box_labels = pd.read_csv('/home/Shelby/blackbird_calls/Dataset_processing/cv4e_calls_channel1_v2.csv')
print(box_labels.columns)
cols_we_need = ["begin_time_s", "end_time_s", "low_freq_hz", "high_freq_hz", "call_type","wav_fname"]
clean_labels = box_labels[cols_we_need]
# rename columns to start_time, end_time, low_f, high_f, annotation, audio_file
clean_labels = clean_labels.rename(columns={
    "begin_time_s": "start_time",
    "end_time_s": "end_time",
    "low_freq_hz": "low_f",
    "high_freq_hz": "high_f",
    "call_type": "annotation",
    "wav_fname": "audio_file"
})
# I need to replace all the spaces with _ in the audio_file column
clean_labels['audio_file'] = clean_labels['audio_file'].str.replace(' ', '_')
# drop any nans in clean_labels
clean_labels = clean_labels.dropna()
# Convert annotation column to object dtype for OpenSoundScape compatibility
clean_labels['annotation'] = clean_labels['annotation'].astype('object')

Index(['Selection', 'View', 'Channel', 'begin_time_s', 'end_time_s',
       'low_freq_hz', 'high_freq_hz', 'delta_time_s', 'call_type',
       'selection_fpath', 'file_id', 'channel_is_one', 'NestID', 'Site',
       'Date', 'Time', 'Model', 'Stage', 'NestAge', 'NestlingAge', 'Trial',
       'Weather', 'wav_fpath', 'wav_fname', 'has_wav', 'has_metadata'],
      dtype='str')


In [9]:
clean_labels["annotation"].value_counts()

annotation
Check    3828
Cheer     707
F         256
E         248
C         202
D         201
Chits     171
B         136
K         130
G          77
I          75
M          56
J          39
L          26
O          24
A          20
N          20
H          14
Growl       5
Name: count, dtype: int64

In [3]:
# calculate how long (start_time to end_time) each F, L and N call is
durations = clean_labels.copy()
durations['duration'] = durations['end_time'] - durations['start_time']
durations_stats = durations.groupby('annotation')['duration'].describe()
print(durations_stats)

             count      mean       std       min       25%       50%  \
annotation                                                             
A             20.0  0.336833  0.043534  0.268248  0.307381  0.328550   
B            136.0  0.201176  0.110469  0.073668  0.126932  0.146092   
C            202.0  0.106212  0.044484  0.036441  0.083822  0.094836   
Check       3828.0  0.101671  0.030694  0.034359  0.081255  0.097148   
Cheer        707.0  0.625307  0.164795  0.068725  0.522595  0.617925   
Chits        171.0  0.756086  0.429869  0.091277  0.453879  0.676371   
D            201.0  0.147284  0.069556  0.049112  0.102023  0.129330   
E            248.0  0.131041  0.073842  0.045805  0.077685  0.110406   
F            256.0  0.094648  0.028812  0.041222  0.068826  0.094901   
G             77.0  0.614858  0.146688  0.326193  0.529466  0.584380   
Growl          5.0  0.251647  0.062536  0.156151  0.220756  0.287390   
H             14.0  0.264234  0.082609  0.162863  0.201048  0.25

In [10]:
# Add full path to audio files
import os

audio_dir = '/mnt/class_data/Shelby/One_Minute_Audio'
clean_labels['audio_file'] = clean_labels['audio_file'].apply(lambda x: os.path.join(audio_dir, x))

# Verify the paths
print(f"Sample paths after update:")
print(clean_labels['audio_file'].head())
annotation = BoxedAnnotations(df=clean_labels)#, audio_files =clean_labels["audio_file"])
from opensoundscape.utils import make_clip_df
clip_df = make_clip_df(files=set(clean_labels["audio_file"]), clip_duration=3.0)
labels = annotation.labels_on_index(clip_df, min_label_overlap=0.2, min_label_fraction=0.5)

Sample paths after update:
0    /mnt/class_data/Shelby/One_Minute_Audio/AZ02_T...
1    /mnt/class_data/Shelby/One_Minute_Audio/AZ02_T...
2    /mnt/class_data/Shelby/One_Minute_Audio/AZ02_T...
3    /mnt/class_data/Shelby/One_Minute_Audio/AZ02_T...
4    /mnt/class_data/Shelby/One_Minute_Audio/AZ02_T...
Name: audio_file, dtype: str


/home/Shelby/miniconda3/envs/BlackbirdOP_py312/lib/python3.12/site-packages/opensoundscape/utils.py:326: FutureWarning: PySoundFile failed. Trying audioread instead.
	Audioread support is deprecated in librosa 0.10.0 and will be removed in version 1.0.
  t = librosa.get_duration(path=path)


In [11]:
# Display the resulting labels dataframe
# Each row is a 3-second clip, columns are call types with 0/1 for absence/presence
print(f"Shape: {labels.shape} (rows = 3s clips, columns = call types)")
print(f"\nCall types found: {list(labels.columns)}")
print(f"\nSample of multi-hot encoded labels:")
labels.head(20)

Shape: (2201, 19) (rows = 3s clips, columns = call types)

Call types found: ['Cheer', 'A', 'Check', 'B', 'K', 'C', 'Chits', 'M', 'Growl', 'D', 'J', 'O', 'I', 'F', 'G', 'H', 'E', 'N', 'L']

Sample of multi-hot encoded labels:


Cheer  \
file                                               start_time end_time          
/mnt/class_data/Shelby/One_Minute_Audio/SL17a_T... 0.0        3.0       False   
                                                   3.0        6.0       False   
                                                   6.0        9.0       False   
                                                   9.0        12.0      False   
                                                   12.0       15.0      False   
                                                   15.0       18.0      False   
                                                   18.0       21.0      False   
                                                   21.0       24.0      False   
                                                   24.0       27.0      False   
                                                   27.0       30.0      False   
                                                   30.0       33.0      False   
                                                   33.0       36.0      False   
                                                   36.0       39.0      False   
                                                   39.0       42.0      False   
                                                   42.0       45.0      False   
                                                   45.0       48.0      False   
                                                   48.0       51.0      False   
                                                   51.0       54.0      False   
                                                   54.0       57.0      False   
                                                   57.0       60.0      False   

                                                                            A  \
file                                               start_time end_time          
/mnt/class_data/Shelby/One_Minute_Audio/SL17a_T... 0.0        3.0       False   
                                                   3.0        6.0       False   
                                                   6.0        9.0       False   
                                                   9.0        12.0      False   
                                                   12.0       15.0      False   
                                                   15.0       18.0      False   
                                                   18.0       21.0      False   
                                                   21.0       24.0      False   
                                                   24.0       27.0      False   
                                                   27.0       30.0      False   
                                                   30.0       33.0      False   
                                                   33.0       36.0      False   
                                                   36.0       39.0      False   
                                                   39.0       42.0      False   
                                                   42.0       45.0      False   
                                                   45.0       48.0      False   
                                                   48.0       51.0      False   
                                                   51.0       54.0      False   
                                                   54.0       57.0      False   
                                                   57.0       60.0      False   

                                                                        Check  \
file                                               start_time end_time          
/mnt/class_data/Shelby/One_Minute_Audio/SL17a_T... 0.0        3.0       False   
                                                   3.0        6.0       False   
                                                   6.0        9.0       False   
                                                   9.0        12.0      False   
                    

In [12]:
# check for duplicate index values in labels
duplicate_indices = labels.index[labels.index.duplicated()].unique()
if len(duplicate_indices) > 0:
    print(f"Found {len(duplicate_indices)} duplicate index values in labels.")
    for idx in duplicate_indices:
        print(f"Duplicate index: {idx}")
else:
    print("No duplicate index values found in labels.")

# drop any duplicate indices, keeping the first occurrence
labels = labels[~labels.index.duplicated(keep='first')] 

No duplicate index values found in labels.


In [13]:
# Check label distribution
print("Number of clips with each call type:")
print(labels.sum().sort_values(ascending=False))
print(f"\nTotal clips: {len(labels)}")
print(f"Clips with at least one call: {(labels.sum(axis=1) > 0).sum()}")
print(f"Clips with no calls: {(labels.sum(axis=1) == 0).sum()}")

Number of clips with each call type:
Check    1081
Cheer     523
Chits     158
E         151
F         150
C         102
D         101
K          89
B          67
G          59
I          49
J          30
M          24
A          18
N          16
L          14
O          14
H          11
Growl       5
dtype: int64

Total clips: 2201
Clips with at least one call: 1815
Clips with no calls: 386


In [14]:
# Filter and rename call types before saving
# Rename call types (merge similar types together)
calltype_mapping = {
    'M': 'A',  # Merge M into A
    'O': 'E',  # Merge O into E
    'I': 'D',  # Merge I into D
    'L': 'C',  # Merge L into C
}

# Apply renaming by combining columns
for old_name, new_name in calltype_mapping.items():
    if old_name in labels.columns and new_name in labels.columns:
        # Merge: if either old or new has a 1, the result should be 1
        labels[new_name] = (labels[old_name] | labels[new_name]).astype(int)
        labels = labels.drop(columns=[old_name])
        print(f"Merged column '{old_name}' into '{new_name}'")
    elif old_name in labels.columns:
        # Just rename if new name doesn't exist
        labels = labels.rename(columns={old_name: new_name})
        print(f"Renamed column '{old_name}' to '{new_name}'")

# Filter out specific call types
call_types_to_exclude = ['F', 'Growl', 'H', 'N']
columns_to_drop = [col for col in call_types_to_exclude if col in labels.columns]
if columns_to_drop:
    labels = labels.drop(columns=columns_to_drop)
    print(f"\nRemoved columns: {columns_to_drop}")

print(f"\nFinal call types: {list(labels.columns)}")
print(f"Call type distribution after filtering:")
print(labels.sum().sort_values(ascending=False))

Merged column 'M' into 'A'
Merged column 'O' into 'E'
Merged column 'I' into 'D'
Merged column 'L' into 'C'

Removed columns: ['F', 'Growl', 'H', 'N']

Final call types: ['Cheer', 'A', 'Check', 'B', 'K', 'C', 'Chits', 'D', 'J', 'G', 'E']
Call type distribution after filtering:
Check    1081
Cheer     523
E         165
Chits     158
D         150
C         111
K          89
B          67
G          59
A          42
J          30
dtype: int64


In [7]:
# Save the multi-hot encoded labels to CSV
output_path = '/home/Shelby/blackbird_calls/Experiments/Detection_of_calls/Datasets/All_call_classes/Split_data_controlled/OpenSoundScape/clip_labels_3s.csv'
labels.to_csv(output_path)
print(f"Saved multi-hot encoded labels to: {output_path}")
print(f"\nThis file contains {len(labels)} rows (3-second clips)")
print(f"and {len(labels.columns)} columns (call types with binary 0/1 labels)")

Saved multi-hot encoded labels to: /home/Shelby/blackbird_calls/Experiments/Detection_of_calls/Datasets/All_call_classes/Split_data_controlled/OpenSoundScape/clip_labels_3s.csv

This file contains 2201 rows (3-second clips)
and 11 columns (call types with binary 0/1 labels)
